# AValidate · Vision baseline (ResNet-18)

Run top to bottom on a **GPU runtime** (Runtime → Change runtime type → GPU: T4 or L4).

Every step caches persistent outputs (checkpoints, results, exported embeddings) on Google Drive under `MyDrive/AValidate/`, so after a disconnect you can reconnect and your models/results are preserved.

**One-time setup:** add a GitHub token with read access to this repo in Colab Secrets (key icon on the left) as `GH_TOKEN`, and switch on notebook access.

In [ ]:
!nvidia-smi -L || echo 'No GPU: switch runtime to GPU (Runtime -> Change runtime type -> T4/L4 GPU)'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, subprocess
from google.colab import userdata

BRANCH = 'feature/video_pipeline'
REPO = '/content/Audio-Video-Confidence'
token = userdata.get('GH_TOKEN')
url = f'https://{token}@github.com/NUS-AIS-GCs/Audio-Video-Confidence.git'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'fetch', '-q', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', '-q', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'reset', '-q', '--hard', f'origin/{BRANCH}'], check=True)
os.chdir(REPO)
!git log --oneline -1

In [ ]:
!pip -q install -r requirements-video.txt
!pip -q install wandb

In [ ]:
# Weights & Biases: reads WANDB_API_KEY from Colab Secrets (key icon). Skipped if secret is missing.
WANDB_ENTITY = 'GC2-AValidate'
try:
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    os.environ['WANDB_ENTITY'] = WANDB_ENTITY
    os.environ.setdefault('WANDB_PROJECT', 'avalidate')
    WANDB = '--wandb'
    print('W&B on:', os.environ['WANDB_ENTITY'] + '/' + os.environ['WANDB_PROJECT'])
except Exception as e:
    WANDB = ''
    print('W&B off:', e)

## 1 · Data verification
Ensures AVE dataset zip exists on Drive (or downloads once), then unzips to fast local scratch disk `/content/work/AVE_Dataset`.

In [ ]:
%cd /content/Audio-Video-Confidence
!python -m vision.data_ave

## 2 · Frame extraction
Extracts 10 frames per video (1 fps, 224×224 RGB) using multithreading directly to the fast local SSD (`/content/work/frames_1fps`).
Takes ~2–3 minutes on Colab.

In [ ]:
%cd /content/Audio-Video-Confidence
!python -m vision.extract_frames

## 3 · Features and split metadata
Prepares `{split}_y.npy`, `{split}_yclip.npy`, and `{split}_ids.csv` in `_local/drive/cache/vision_features_v1` on Drive.

In [ ]:
%cd /content/Audio-Video-Confidence
!python -m vision.features

## 4 · Sanity check (Overfit 16 clips)
Verifies loss convergence and pipeline health in under 1 minute.

In [ ]:
%cd /content/Audio-Video-Confidence
!python -m vision.train --run-id overfit --backbone cnn --overfit 16 --epochs 20 --batch 4

## 5 · Full Vision Baseline Training
Trains pretrained ResNet-18 on all 3,339 training clips for 30 epochs with cosine LR decay, AMP fp16, and class-weighted loss (`--balance sqrt`).
Best checkpoint (by validation per-second macro-F1) is saved to Drive: `MyDrive/AValidate/artifacts/vision_baseline/{RUN_ID}/best.pt`.

In [ ]:
%cd /content/Audio-Video-Confidence
RUN_ID = 'vision_v1'
WANDB = globals().get('WANDB', '')
if not WANDB: print('W&B is OFF: run the W&B cell first if you want tracking')
!python -m vision.train --run-id {RUN_ID} --backbone resnet18 --epochs 30 --batch 16 --lr 5e-4 --balance sqrt {WANDB}

## 6 · Test Set Evaluation
Evaluates on the official AVE test split (402 clips), computes macro-F1 and accuracy, saves `results.json`, and displays the confusion matrix.

In [ ]:
%cd /content/Audio-Video-Confidence
!python -m vision.evaluate --run-id {RUN_ID} {WANDB}
from IPython.display import Image, display
confusion_path = f'/content/drive/MyDrive/AValidate/artifacts/vision_baseline/{RUN_ID}/confusion_test.png'
if os.path.exists(confusion_path):
    display(Image(confusion_path))

## 7 · Export Embeddings for Multimodal Fusion
Exports 10-second per-clip visual embeddings `(N, 10, 512)` float16 to:
`MyDrive/AValidate/embeddings/vision_baseline_v1/{split}_emb.npy`.

In [ ]:
%cd /content/Audio-Video-Confidence
!python -m vision.export_embeddings --run-id {RUN_ID}